In [ ]:
import mne
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from mne.datasets.sleep_physionet.age import fetch_data
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
# Data loading and feature extraction

stage_names = {
    'Wake': 'Sleep stage W',
    'N1':   'Sleep stage 1',
    'N2':   'Sleep stage 2',
    'N3':   'Sleep stage 3',
    'REM':  'Sleep stage R',
}

def extract_features_per_second(signal, fs, n_steps=30):
    # Split 30s epoch into n_steps segments, extract FFT features per segment
    # Returns shape (n_steps, 6)
    step_len = len(signal) // n_steps
    features_seq = []

    for i in range(n_steps):
        seg = signal[i * step_len : (i+1) * step_len]
        fft_result = np.fft.rfft(seg)
        powers = np.abs(fft_result) ** 2
        freqs  = np.fft.rfftfreq(len(seg), d=1/fs)

        delta = np.sum(powers[(freqs >= 0.5) & (freqs < 4)])
        theta = np.sum(powers[(freqs >= 4)   & (freqs < 8)])
        alpha = np.sum(powers[(freqs >= 8)   & (freqs < 13)])
        beta  = np.sum(powers[(freqs >= 13)  & (freqs < 30)])
        theta_beta  = theta / (beta  + 1e-6)
        delta_theta = delta / (theta + 1e-6)

        features_seq.append([delta, theta, alpha, beta, theta_beta, delta_theta])

    return np.array(features_seq)

all_X = []
all_y = []

for subject_id in [0, 1, 2, 3, 4]:
    files    = fetch_data(subjects=[subject_id], recording=[1])
    psg_file = files[0][0]
    hyp_file = files[0][1]
    raw         = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
    fs_real     = int(raw.info['sfreq'])
    data, _     = raw['EEG Fpz-Cz']
    annotations = mne.read_annotations(hyp_file)

    for stage_label, stage_raw in stage_names.items():
        for j in range(len(annotations)):
            if annotations.description[j] == stage_raw:
                onset     = annotations.onset[j]
                start_idx = int(onset * fs_real)
                end_idx   = int((onset + 30) * fs_real)
                if end_idx <= data.shape[1]:
                    epoch    = data[0, start_idx:end_idx]
                    features = extract_features_per_second(epoch, fs_real)
                    all_X.append(features)
                    all_y.append(stage_label)

X = np.array(all_X)  # (668, 30, 6)
y = np.array(all_y)

print(f"X shape: {X.shape}")
print(f"Label distribution: { {s: np.sum(y==s) for s in stage_names} }")

In [ ]:
# Label encoding, train/test split, normalization, tensor conversion

le = LabelEncoder()
y_enc = le.fit_transform(y)
print(f"Class order: {le.classes_}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

n_train, n_steps, n_feat = X_train.shape
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, n_feat)).reshape(n_train, n_steps, n_feat)
X_test_scaled  = scaler.transform(X_test.reshape(-1, n_feat)).reshape(-1, n_steps, n_feat)

X_train_t = torch.FloatTensor(X_train_scaled)
X_test_t  = torch.FloatTensor(X_test_scaled)
y_train_t = torch.LongTensor(y_train)
y_test_t  = torch.LongTensor(y_test)

print(f"Train tensor: {X_train_t.shape}")
print(f"Test tensor:  {X_test_t.shape}")

In [ ]:
# Model definition

class SleepLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

In [ ]:
# Training

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

counts = np.array([152, 225, 148, 63, 80])  # N1 N2 N3 REM Wake
class_weights = torch.FloatTensor(1.0 / counts)
class_weights = class_weights / class_weights.sum() * 5

model = SleepLSTM(input_size=6, hidden_size=64, num_layers=2, num_classes=5)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

best_acc = 0
best_epoch = 0
no_improve = 0
patience = 20

for epoch in range(200):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            preds = model(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
    acc = correct / len(test_ds)

    scheduler.step()

    if acc > best_acc:
        best_acc = acc
        best_epoch = epoch + 1
        no_improve = 0
        torch.save(model.state_dict(), 'best_lstm.pt')
    else:
        no_improve += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} : Loss: {total_loss/len(train_loader):.4f} , Acc: {acc:.2%}")

    if no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

model.load_state_dict(torch.load('best_lstm.pt'))
print(f"\nBest accuracy: {best_acc:.2%} (Epoch {best_epoch})")

In [ ]:
# Evaluation

model_final = SleepLSTM(input_size=6, hidden_size=64, num_layers=2, num_classes=5)
model_final.load_state_dict(torch.load('best_lstm.pt'))
model_final.eval()

all_preds = []
all_true  = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model_final(X_batch).argmax(dim=1)
        all_preds.extend(preds.numpy())
        all_true.extend(y_batch.numpy())

cm = confusion_matrix(all_true, all_preds)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(cmap='Purples')
plt.title('LSTM Confusion Matrix')
plt.show()

print(classification_report(all_true, all_preds, target_names=le.classes_))